In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")

train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [5]:
train.health_condition.unique()

array(['unhealthy', 'at-risk', 'fit'], dtype=object)

In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
train.groupby('health_condition')['id'].count()

health_condition
at-risk      592561
fit           39803
unhealthy     57724
Name: id, dtype: int64

In [9]:
x, y = train.drop(columns=['id', 'health_condition'], axis=1), train[['health_condition']]
x.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [10]:
y.head()

,health_condition
0,unhealthy
1,at-risk
2,unhealthy
3,unhealthy
4,at-risk


In [11]:
cat_features = x.columns[np.where(x.dtypes != float)[0]].values.tolist()
print(cat_features)

# make sure that the categorical features are encoded as strings
x[cat_features] = x[cat_features].astype(str)

['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [12]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
x_test.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
304516,6.82,67.9,27.17,2464.0,13456.0,39.9,2.05,veg,medium,good,moderate,no,other
165358,7.97,92.9,16.86,NaN,7456.0,26.9,2.14,balanced,high,good,moderate,no,female
671841,NaN,NaN,22.18,2352.0,4140.0,19.7,2.33,non-veg,low,poor,sedentary,occasional,male
403460,NaN,83.1,21.93,2620.0,11656.0,41.1,1.22,non-veg,medium,good,moderate,occasional,other
351133,8.14,80.0,23.41,2442.0,4484.0,38.9,2.32,veg,medium,nan,sedentary,no,other


In [14]:
model = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    task_type = 'GPU',
    devices = '0'
)
model.fit(x_train, y_train, cat_features=cat_features)

0:	learn: 0.9660086	total: 6.49s	remaining: 10m 42s
1:	learn: 0.8655135	total: 6.5s	remaining: 5m 18s
2:	learn: 0.7861504	total: 6.52s	remaining: 3m 30s
3:	learn: 0.7215640	total: 6.53s	remaining: 2m 36s
4:	learn: 0.6680332	total: 6.54s	remaining: 2m 4s
5:	learn: 0.6232842	total: 6.55s	remaining: 1m 42s
6:	learn: 0.5852682	total: 6.57s	remaining: 1m 27s
7:	learn: 0.5528050	total: 6.58s	remaining: 1m 15s
8:	learn: 0.5246116	total: 6.59s	remaining: 1m 6s
9:	learn: 0.5002170	total: 6.61s	remaining: 59.5s
10:	learn: 0.4549014	total: 6.62s	remaining: 53.6s
11:	learn: 0.4172772	total: 6.63s	remaining: 48.6s
12:	learn: 0.3855128	total: 6.64s	remaining: 44.5s
13:	learn: 0.3583159	total: 6.66s	remaining: 40.9s
14:	learn: 0.3343723	total: 6.67s	remaining: 37.8s
15:	learn: 0.3137364	total: 6.68s	remaining: 35.1s
16:	learn: 0.2954340	total: 6.7s	remaining: 32.7s
17:	learn: 0.2794770	total: 6.71s	remaining: 30.6s
18:	learn: 0.2651512	total: 6.72s	remaining: 28.7s
19:	learn: 0.2526537	total: 6.73s	r

CatBoostClassifier(depth=6, devices='0', iterations=100, learning_rate=0.1, loss_function='MultiClass', task_type='GPU')

In [15]:
y_pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.966076888521787


In [16]:
final_model = model = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    task_type = 'GPU',
    devices = '0'
)
final_model.fit(x, y, cat_features=cat_features)

0:	learn: 0.9659830	total: 14.3ms	remaining: 1.42s
1:	learn: 0.8654123	total: 27.9ms	remaining: 1.37s
2:	learn: 0.7861093	total: 41.2ms	remaining: 1.33s
3:	learn: 0.7214788	total: 54.1ms	remaining: 1.3s
4:	learn: 0.6679443	total: 68.5ms	remaining: 1.3s
5:	learn: 0.6232460	total: 81.4ms	remaining: 1.27s
6:	learn: 0.5850281	total: 94.2ms	remaining: 1.25s
7:	learn: 0.5527049	total: 107ms	remaining: 1.23s
8:	learn: 0.5245660	total: 120ms	remaining: 1.21s
9:	learn: 0.5000658	total: 132ms	remaining: 1.19s
10:	learn: 0.4548162	total: 145ms	remaining: 1.18s
11:	learn: 0.4171568	total: 159ms	remaining: 1.17s
12:	learn: 0.3856770	total: 173ms	remaining: 1.16s
13:	learn: 0.3585130	total: 186ms	remaining: 1.14s
14:	learn: 0.3348320	total: 199ms	remaining: 1.13s
15:	learn: 0.3139277	total: 212ms	remaining: 1.11s
16:	learn: 0.2959840	total: 226ms	remaining: 1.1s
17:	learn: 0.2798883	total: 239ms	remaining: 1.09s
18:	learn: 0.2652146	total: 252ms	remaining: 1.07s
19:	learn: 0.2526426	total: 265ms	rem

CatBoostClassifier(depth=6, devices='0', iterations=100, learning_rate=0.1, loss_function='MultiClass', task_type='GPU')

In [17]:
final_test = test.drop(columns=['id'], axis=1)
final_test = final_test['water_intake'].fillna(2.19)
pred = final_model.predict(final_test)

submission = pd.DataFrame({
    "id": test["id"],
    "health_condition": pred
})

# Save to CSV without the index
submission.to_csv('submission.csv', index=False)
print("Submission file saved successfully!")

CatBoostError: Invalid type for cat_feature[non-default value idx=0,feature_idx=7]=1.14 : cat_features must be integer or string, real number values and NaN values should be converted to string.

In [22]:
final_test.iloc[6:8]

6    2.01
7    1.14
Name: water_intake, dtype: float64

In [ ]:
test['water_intake'].mean()

In [ ]:
test[cat_features] = test[cat_features].astype(str)

pred = final_model.predict(test.drop(columns=['id'], axis=1))

submission = pd.DataFrame({
    "id": test["id"],
    "health_condition": pred
})

# Save to CSV without the index
submission.to_csv('submission.csv', index=False)
print("Submission file saved successfully!")

In [ ]:
test[cat_features] = test[cat_features].astype(str)
test.info()

In [ ]:
cat_features

In [ ]:
test[cat_features].astype(str).info()